# M5 Walmart Demand Forecasting — Phase 3: Business Impact Analysis

**Question:** How much money does better forecasting save?  
**Method:** Simulate inventory costs under naive baseline vs. best ML model.  


---
## 1. Setup & Load Results


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.data_loader import load_ca1_daily

df = load_ca1_daily()

# Load model metrics from Phase 2
metrics_path = "../outputs/metrics/model_comparison.csv"
if os.path.exists(metrics_path):
    metrics = pd.read_csv(metrics_path, index_col="model")
    print(metrics)
else:
    print("Run 02_models.ipynb first to generate model_comparison.csv")

---
## 2. Business Assumptions


In [ ]:
# Cost parameters (adjust to match Walmart CA_1 realistic estimates)
AVG_ITEM_PRICE     = 8.50   # USD — average selling price per unit
HOLDING_COST_RATE  = 0.25   # 25% of item value per year = holding cost
STOCKOUT_COST_RATE = 1.10   # lost profit per unit (1.1x selling price)
SAFETY_STOCK_DAYS  = 3      # days of buffer stock

daily_avg_sales = df["sales"].mean()
daily_holding_cost_per_unit = (AVG_ITEM_PRICE * HOLDING_COST_RATE) / 365

print(f"Average daily sales (CA_1): {daily_avg_sales:,.0f} units")
print(f"Daily holding cost per unit: ${daily_holding_cost_per_unit:.4f}")

---
## 3. Naive Baseline: Forecast = Last 28-Day Average


In [ ]:
TEST_DAYS = 28
train = df.iloc[:-TEST_DAYS].copy()
test  = df.iloc[-TEST_DAYS:].copy()
y_test = test["sales"].values

# Naive forecast: rolling mean of last 28 days of training
naive_pred = np.full(TEST_DAYS, train["sales"].tail(28).mean())

naive_errors = y_test - naive_pred
naive_overstock  = naive_errors[naive_errors < 0]   # over-predicted → excess inventory
naive_stockout   = naive_errors[naive_errors > 0]   # under-predicted → missed sales

naive_holding_cost  = abs(naive_overstock.sum()) * daily_holding_cost_per_unit
naive_stockout_cost = abs(naive_stockout.sum())  * AVG_ITEM_PRICE * STOCKOUT_COST_RATE
naive_total_cost    = naive_holding_cost + naive_stockout_cost

print(f"Naive baseline — 28-day cost:")
print(f"  Holding cost : ${naive_holding_cost:,.2f}")
print(f"  Stockout cost: ${naive_stockout_cost:,.2f}")
print(f"  Total        : ${naive_total_cost:,.2f}")

---
## 4. Best Model Cost Simulation


In [ ]:
# Load best model predictions from outputs (replace with actual predictions)
# Example: load Prophet or LSTM predictions saved in Phase 2
# best_preds = np.load("../outputs/metrics/best_model_preds.npy")

# PLACEHOLDER — replace with real preds after Phase 2
# Simulate 30% MAPE improvement over naive for demonstration
noise_scale = naive_pred.std() * 0.5
best_preds = y_test + np.random.normal(0, noise_scale, TEST_DAYS)
best_preds = np.maximum(best_preds, 0)

best_errors = y_test - best_preds
best_overstock  = best_errors[best_errors < 0]
best_stockout   = best_errors[best_errors > 0]

best_holding_cost  = abs(best_overstock.sum()) * daily_holding_cost_per_unit
best_stockout_cost = abs(best_stockout.sum())  * AVG_ITEM_PRICE * STOCKOUT_COST_RATE
best_total_cost    = best_holding_cost + best_stockout_cost

print(f"Best model — 28-day cost:")
print(f"  Holding cost : ${best_holding_cost:,.2f}")
print(f"  Stockout cost: ${best_stockout_cost:,.2f}")
print(f"  Total        : ${best_total_cost:,.2f}")

---
## 5. Savings Summary & Annualisation


In [ ]:
savings_28d = naive_total_cost - best_total_cost
savings_annual = savings_28d * (365 / TEST_DAYS)
pct_improvement = (savings_28d / naive_total_cost) * 100

print(f"\n{'='*45}")
print(f"  28-day savings  : ${savings_28d:>10,.2f}")
print(f"  Annual savings  : ${savings_annual:>10,.2f}")
print(f"  Improvement     : {pct_improvement:.1f}%")
print(f"{'='*45}")

In [ ]:
# Bar chart: cost breakdown comparison
labels = ["Holding Cost", "Stockout Cost", "Total Cost"]
naive_vals = [naive_holding_cost, naive_stockout_cost, naive_total_cost]
best_vals  = [best_holding_cost,  best_stockout_cost,  best_total_cost]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, naive_vals, width, label="Naive Baseline", color="tomato")
ax.bar(x + width/2, best_vals,  width, label="Best ML Model",  color="steelblue")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Cost (USD) — 28 Days")
ax.set_title("Inventory Cost: Naive Baseline vs Best ML Model — CA_1")
ax.legend()
plt.tight_layout()
fig.savefig("../outputs/plots/business_impact.png", dpi=150)
plt.show()

---
## 6. Safety Stock Optimisation


In [ ]:
# Safety stock = Z * σ_forecast * √lead_time
# Z = 1.645 for 95% service level
Z = 1.645
LEAD_TIME_DAYS = 3
forecast_std_naive = np.std(naive_errors)
forecast_std_best  = np.std(best_errors)

safety_stock_naive = Z * forecast_std_naive * np.sqrt(LEAD_TIME_DAYS)
safety_stock_best  = Z * forecast_std_best  * np.sqrt(LEAD_TIME_DAYS)
safety_stock_savings_units = safety_stock_naive - safety_stock_best
safety_stock_savings_cost  = safety_stock_savings_units * daily_holding_cost_per_unit * 365

print(f"Safety stock (95% SL, {LEAD_TIME_DAYS}-day lead time):")
print(f"  Naive baseline: {safety_stock_naive:,.0f} units")
print(f"  Best model    : {safety_stock_best:,.0f} units")
print(f"  Reduction     : {safety_stock_savings_units:,.0f} units")
print(f"  Annual holding cost saving: ${safety_stock_savings_cost:,.2f}")

---
## 7. Business Impact Summary

| Metric | Naive Baseline | Best ML Model | Improvement |
|---|---|---|---|
| 28-day total cost | — | — | — |
| Annualised savings | — | — | — |
| Safety stock (units) | — | — | — |
| Safety stock annual saving | — | — | — |

*(Fill in after completing the simulation with real model predictions.)*

**Key takeaway:** Even a modest improvement in MAPE (e.g., 5 percentage points) translates into meaningful dollar savings at CA_1 scale — and this is one store out of Walmart's ~5,000 US locations.
